<img src="https://raw.githubusercontent.com/PrimeReSolutions/python_actuarios/main/datos/img/logo_curso.png" width="450">

<p style="font-family: Arial; color: navy; text-align: center; font-size: 13px; letter-spacing: 1px; text-transform: uppercase; margin-bottom: 0;">
WSP — Python aplicado a modelos actuariales
</p>

<h1 style="background-color:#0070C0; color:white; text-align:center; font-family:Arial; padding:18px 0; border-radius:6px; margin-top:6px;">
Capstone — Caso integrador
</h1>

<div style="outline: 2px solid #EFB400; color:#000000; font-family: Arial; padding: 14px 18px; border-radius: 6px; margin-top: 14px;">
<h3 style="margin-top:0;">🎯 Objetivos</h3>
<ul>
<li>Integrar en un solo flujo las etapas trabajadas en el curso: calidad de datos (<b>Sesión 2</b>), triángulos e IBNR (<b>Sesión 3</b>), reservas de vida (<b>Sesión 4</b>), simulación y riesgo de prima (<b>Sesión 5</b>) y riesgo de reserva / solvencia (<b>Sesión 6</b>).</li>
<li>Construir el balance simplificado de una aseguradora y calcular la reserva de primas no devengadas (RPND).</li>
<li>Estimar los riesgos de mortalidad, longevidad, reserva y prima, y agregarlos en un requerimiento de capital de solvencia (SCR).</li>
<li>Obtener el ratio de solvencia final de la compañía.</li>
</ul>
</div>

<div style="outline: 2px solid #0070C0; color:#000000; font-family: Arial; padding: 14px 18px; border-radius: 6px; margin-top: 12px;">
<h3 style="margin-top:0;">✍️ Cómo usar este notebook</h3>
<p>Los huecos que debes completar están marcados con <code>***</code>. Reemplázalos por el código o valor correspondiente y ejecuta la celda. El notebook <b>solucionario</b> (<code>solucionarios/capstone_ejercicio_final_solucionario.ipynb</code>) tiene la respuesta completa de cada <code>***</code>, con la misma estructura de celdas que este notebook. Ejecuta las celdas en orden: varias reutilizan variables definidas en celdas anteriores.</p>
</div>


<h1 style="outline: 2px solid #EFB400; color: #000000; text-align: left; font-family: Arial; padding: 12px; border-radius: 6px;">
📚 Librerías
</h1>

### ⚙️ Celda de arranque

Si trabajas en **Google Colab**, ejecuta esta celda **antes que cualquier otra**: descarga los datos del curso desde GitHub y deja el notebook listo para leerlos. Vuelve a ejecutarla cada vez que Colab reinicie el entorno. En tu PC (instalación local) no hace nada.

In [ ]:
# ⚙️ Celda de arranque: prepara el entorno en Google Colab (en tu PC no hace nada)
import os, sys

if "google.colab" in sys.modules:
    if not os.path.exists("/content/python_actuarios"):
        !git clone -q --depth 1 https://github.com/PrimeReSolutions/python_actuarios.git /content/python_actuarios
    !pip install -q chainladder==0.8.24 sparse==0.14.0
    %cd /content/python_actuarios/notebooks

In [ ]:
import numpy as np
import pandas as pd
import datetime
import chainladder as cl
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
pd.options.display.float_format = '{:,.2f}'.format

DATOS = Path("../datos") if Path("../datos").exists() else Path("datos")

<h1 style="outline: 2px solid #EFB400; color: #000000; text-align: left; font-family: Arial; padding: 12px; border-radius: 6px;">
📚 Parámetros
</h1>

In [ ]:
fecha_calculo = datetime.datetime.strptime('2024-09-30', '%Y-%m-%d')

---
# 📝 1. Balance
---


In [ ]:
data = {'Concepto': ['Caja Banco', 'Bonos', 'Inmuebles', 'Reservas de riesgos en curso', 'Reservas de siniestros', 'Reservas matemáticas', 'Cuentas por pagar'],
    'Monto ($)': [1000000, 45000000, 10000000, '-', '-', '-', 500000]}

# Creando el DataFrame
balance_general = pd.DataFrame(data)
balance_general

---
# 📝 2. Calidad de datos
---

> 🔗 **Conexión:** Esta limpieza de fechas, edades y vigencias es la misma que trabajaste en la Sesión 2 (Calidad de datos).


#### **Aplica los siguientes criterios a la siguiente base de datos:**
- **1. Identifica** fechas menores a 1910 (**consistencia de fechas**)
- **2. Identifica** pólizas pertenecientes a **menores de edad**
- **3. Identifica pólizas no vigentes** (F. inicio <= F. calculo < F. fin)
- **4. Identifica montos vacíos**
- **5. Identifica montos negativos**

<details>
<summary>💡 Pista 1 (concepto)</summary>

Este es el mismo flujo que trabajaste en la **Sesión 2 (Calidad de datos)**: importar el Excel, dejar las fechas y los montos con el tipo de dato correcto, y luego aplicar los 5 criterios de depuración (fechas anteriores a 1910, asegurados menores de edad, pólizas no vigentes, montos vacíos y montos negativos) guardando cada grupo de pólizas problemáticas por separado para poder consolidarlas al final en un solo resumen de incidencias.

<details>
<summary>💡 Pista 2 (código)</summary>

Piensa en: `pd.read_excel(ruta)`, `pd.to_datetime(..., format=..., errors='coerce')`, `.astype(float)`, `.str.contains(...)` para ubicar columnas por nombre, máscaras booleanas combinando condiciones con `&` y `|` (cada condición entre paréntesis), `.isnull()`, `pd.concat([...], axis=0, ignore_index=True)` para juntar los distintos `error*`, y `.groupby([...]).agg({...: 'count', ...: 'sum'})` para el resumen final.

</details>
</details>


1️⃣ Ruta del archivo

In [ ]:
ruta_rmat = DATOS / "rmat_ejercicio_final.xlsx"

2️⃣ Importar el archivo

In [ ]:
base_rmat_ori = pd.read_excel(***)

In [ ]:
base_rmat_ori.head()

3️⃣ Aplicar formatos

##### Formato fecha 📅

In [ ]:
base_rmat = base_rmat_ori.copy()
base_rmat['fecha nacimiento'] = pd.to_datetime(base_rmat['fecha nacimiento'], format = ***, errors="coerce")
base_rmat['fecha inicio'] = pd.to_datetime(base_rmat['fecha inicio'], format = ***, errors="coerce")
base_rmat['fecha fin'] = pd.to_datetime(base_rmat['fecha fin'], format = ***, errors="coerce")

##### Formato numérico 🔢

In [ ]:
base_rmat['gastos_adm'] = base_rmat['gastos_adm'].astype(float)
base_rmat[('Prima')] = base_rmat['Prima'].astype(float)

colSA = base_rmat.columns[base_rmat.columns.str.contains(pat = ***)]
base_rmat[colSA] = base_rmat[***].apply(pd.to_numeric, errors='coerce')

⚠️ **1. Identificar fechas menores a 1910**

In [ ]:
#Definimos una fecha mínima como referencia para asegurar la coherencia temporal de los datos
fecha_minima =  datetime.datetime.strptime(***, '%Y-%m-%d')

#Creamos una copia de la base de pólizas
rm1 = base_rmat.copy()

#Filtramos las pólizas que tienen fechas coherentes
rm2 = rm1[(rm1['fecha nacimiento'] > fecha_minima) *** (rm1['fecha inicio'] > fecha_minima) *** (rm1['fecha fin'] > fecha_minima)]

In [ ]:
#Identificamos las pólizas con fechas inconsistentes y las guardamos en el df "error1"
error1 = rm1[(rm1['fecha nacimiento'] *** fecha_minima) | (rm1['fecha inicio'] *** fecha_minima) | (rm1['fecha fin'] *** fecha_minima)].copy()
error1['tipo'] = 'menor a 1910'
error1

⚠️ **2. Identifica pólizas pertenecientes a menores de edad**

In [ ]:
#Definimos la edad mínima para estar asegurado
limite_edad = fecha_calculo - datetime.timedelta(days=***)

#Creamos una copia de la base de pólizas que no tienen fechas inconsistentes
rm3 = rm2.copy()

#Filtramos las pólizas que tienen asegurados mayores de 18 años
rm4 = rm3[(rm3['fecha nacimiento'] *** limite_edad)]

In [ ]:
#Identificamos las pólizas con asegurados menores de edad y las guardamos en el df "error2"
error2 = rm3[(rm3['fecha nacimiento'] > limite_edad)].copy()
error2['tipo'] = 'menor de edad'
error2

⚠️ **3. Identifica pólizas no vigentes**

In [ ]:
#Filtramos las pólizas vigentes
rm5 = rm4.copy()
rm6 = rm5[(rm5['fecha inicio'] *** fecha_calculo) & (fecha_calculo *** rm5['fecha fin'])]

In [ ]:
#Identificamos las pólizas no vigentes y las guardamos en el df "error3"
error3 = rm5[(rm5['fecha inicio'] > fecha_calculo) | (fecha_calculo >= rm5['fecha fin'])].copy()
error3['tipo'] = 'no vigente'
error3

⚠️ **4. Identifica montos vacíos**

In [ ]:
#Creamos una copia de la base de pólizas con fechas consistentes, vigentes
rm7 = rm6.copy()

#Filtramos las pólizas que no tienen valores vacíos en las columnas "Prima" y "suma asegurada"
rm8 = rm7[~ ( (rm7['Prima'].***) | (rm7['suma asegurada'].***) ) ]

In [ ]:
#Identificamos las pólizas con vacíos y las guardamos en el df "error4"
error4 = rm7[(rm7['Prima'].isnull()) *** (rm7['suma asegurada'].isnull())].copy()  # base con errores
error4['tipo'] = 'montos vacios'
error4

⚠️ **5. Identifica montos negativos**

In [ ]:
#Creamos una copia de la base de pólizas con fechas consistentes, vigentes y sin vacios
rm9= rm8.copy()

#Filtramos las pólizas que tienen valores positivos en las columnas "Prima" y "suma_asegurada"
rm10 = rm9[(rm9['Prima'] > 0) *** (rm9['suma asegurada'] > 0)]

In [ ]:
#Identificamos las pólizas con montos negativos y las guardamos en el df "error5"
error5 = rm9[(rm9['Prima'] <= 0) | (rm9['suma asegurada'] <= 0)].copy()
error5['tipo'] = 'monto negativo'
error5

#### **📊 Revisión de errores**

In [ ]:
#Agrupamos los 4 grupos de pólizas depuradas
error = pd.concat([***, ***, ***, ***, ***], axis=***, ignore_index = True)

#Creamos una tabla con el número de pólizas y monto de primas según nombre_producto
resumen_incidencias = error.groupby([***, 'tipo']).agg({'num_poliza':***, 'suma asegurada':'sum'})
resumen_incidencias = resumen_incidencias.reset_index(1)

total = base_rmat['suma asegurada'].sum(axis=0)  #suma de SA del archivo rmat

#Ponderamos las incidencias según el monto de SA
resumen_incidencias['impacto (%)'] = (resumen_incidencias['suma asegurada'] / total) * 100

resumen_incidencias.loc[resumen_incidencias['impacto (%)'] <= 5, 'impacto'] = 'leve'
resumen_incidencias.loc[(resumen_incidencias['impacto (%)'] > 5) & (resumen_incidencias['impacto (%)'] <= 20), 'impacto'] = 'moderado'
resumen_incidencias.loc[resumen_incidencias['impacto (%)'] > 20, 'impacto'] = 'grave'

resumen_incidencias.columns = [ 'Error', 'Certificados', 'Suma asegurada', 'impacto (%)', 'impacto']
resumen_incidencias.loc['total'] = resumen_incidencias.select_dtypes(include='number').sum()

pd.options.display.float_format = '{:,.2f}'.format
resumen_incidencias

---
# 📝 3. Riesgo de mortalidad
---

> 🔗 **Conexión:** Las reservas matemáticas y los escenarios de estrés de mortalidad/longevidad son los que viste en la Sesión 4 (Reservas de vida).


  
  
**Calcule el riesgo de mortalidad para el producto trabajado en el ejercicio anterior**
- Asuma un incremento del 15%


<details>
<summary>💡 Pista 1 (concepto)</summary>

Igual que en la **Sesión 4 (Reservas de vida)**: construyes los triángulos de mortalidad (qx) y caída (cx) por póliza, armas los flujos nominales (beneficio, gastos, primas), los "probabilizas" con las probabilidades de sobrevivencia/mortalidad/caída, los descuentas financieramente y sumas para obtener la reserva base. Después repites **todo** el proceso subiendo la mortalidad un 15% para el escenario de estrés, y la diferencia entre ambos escenarios es el riesgo de mortalidad.

<details>
<summary>💡 Pista 2 (código)</summary>

Piensa en: selección de columnas con `tabla[[col1, col2]]`, recorte de matrices con `.iloc[:, a:b]`, multiplicación elemento a elemento entre DataFrames del mismo tamaño (`*`), `pd.concat([...], axis=1)` para repetir una columna y armar una matriz, `.cumprod(axis=1)` para la sobrevivencia acumulada, y `.sum(axis=1)` / `.sum()` para totalizar flujos y llegar a la reserva.

</details>
</details>


1️⃣ Datos

In [ ]:
rmat = ***.copy()

- Considere la siguiente tabla de mortalidad y vector de tasas de descuento

In [ ]:
ruta_tabla = DATOS / "tabla_qx.xlsx"

In [ ]:
ruta_vtd = DATOS / "vector_tasas_descuento.xlsx"

➡️ Importar archivos

In [ ]:
tabla_qx = pd.read_excel(ruta_tabla)
vtd = pd.read_excel(ruta_vtd)

2️⃣ Tablas de mortalidad

In [ ]:
tabla_MP = tabla_qx[[***, ***]]

#-------------------------------------------------------------------------------------------------
# 1.Indices del vector con forma de matriz
#-------------------------------------------------------------------------------------------------
tabla = tabla_MP.copy()
n = tabla.shape[0]
c = np.array((range(0,n)))[::-1]
def aux_AP(c): return np.array(range(0,c+1))
AP = list(map(aux_AP,c))
AP = np.concatenate(AP)
DP  = np.repeat(range(0,n+1),(range(0,n+1)[::-1]))
CY = AP + DP

#-------------------------------------------------------------------------------------------------
# 2. Flujos a partir de un vector: qx
#-------------------------------------------------------------------------------------------------
qx_vector = tabla.***[CY].to_numpy()

n = tabla.shape[0]
triangle_m = np.zeros((n, n))

f = 0
for col in range(0,n):
    for fila in range(0,n):
        triangle_m[fila][col] = qx_vector[f]
        f = f+1
    n=n-1

triangle_m = pd.DataFrame(triangle_m)
triangle_m.insert(0, ***, np.arange(0, triangle_m.shape[0]))

#-------------------------------------------------------------------------------------------------
# 3. Flujos a partir de un vector: cx
#-------------------------------------------------------------------------------------------------
tabla2 = tabla_qx[[***, ***]]

cx_vector = tabla2.***[CY].to_numpy()

n = tabla2.shape[0]
triangle_c = np.zeros((n, n))

f = 0
for col in range(0,n):
    for fila in range(0,n):
        triangle_c[fila][col] = cx_vector[f]
        f = f+1
    n=n-1

triangle_c = pd.DataFrame(triangle_c)
triangle_c.insert(0, 't', np.arange(0, triangle_c.shape[0]))

3️⃣ Información de las pólizas

In [ ]:
#-------------------------------------------------------------------------------------------------
# 1. Cálculo de variables
#-------------------------------------------------------------------------------------------------
rmat['vigencia'] = np.ceil((pd.to_datetime(rmat[***]) - pd.to_datetime(rmat[***])).dt.days / 365.25).astype(int)
rmat['t_poliza'] = np.ceil((pd.to_datetime(fecha_calculo) - pd.to_datetime(rmat[***])).dt.days / 365.25).astype(int)
rmat['Edad_actuarial'] = (fecha_calculo.year - pd.DatetimeIndex(rmat['fecha nacimiento']).year
                          + (fecha_calculo.month - pd.DatetimeIndex(rmat['fecha nacimiento']).month) / 12
                          + (fecha_calculo.day - pd.DatetimeIndex(rmat['fecha nacimiento']).day) / 365.25).astype(int)
rmat['periodo restante'] = (rmat[***] - rmat[***]).astype(int) + 1
rmat['monto gasto'] = rmat['Prima'] * rmat[***]


#-------------------------------------------------------------------------------------------------
# 2. Matriz auxiliar del período restante de vigencia
#-------------------------------------------------------------------------------------------------

# Creación de la matriz
matriz_polizas = np.zeros((rmat.shape[0], max(rmat['periodo restante'])))
matriz_polizas[:,0] = rmat[***]
ncp = matriz_polizas.shape[1]

#Según edad actuarial
def repet (matriz_polizas): return np.concatenate((np.repeat(1,matriz_polizas[0]),
                                                   np.repeat(0,ncp-matriz_polizas[0])),axis=0)
mat_zer = pd.DataFrame(list(map(repet,matriz_polizas)))
mat_zer.insert(0,'Edad',rmat['Edad_actuarial'])
mat_zer.insert(1, 't_res', rmat['periodo restante'])
mat_zer.insert(2, 'num_poliza', rmat['num_poliza'])

#Según t póliza
mat_zer2 = mat_zer.iloc[:, 3:mat_zer.shape[1]]
mat_zer2.insert(0,'t', rmat[***])
mat_zer2.insert(1, 'num_poliza', rmat['num_poliza'])

#-------------------------------------------------------------------------------------------------
# 3. Construcción de una matriz de probabilidades para cada asegurado
#-------------------------------------------------------------------------------------------------

# Unión de dos bases de datos a través de una llave: qx
mortalidad = triangle_m.iloc[:,0:(max(rmat['periodo restante'])+1)]
aux_qx = pd.merge(mat_zer, mortalidad,on='Edad',how='left')

qx1 = (aux_qx.iloc[:,3:mat_zer.shape[1]])
qx1.rename(columns=lambda x: x.replace('_x', ''), inplace=True)
qx2 = (aux_qx.iloc[:,(mat_zer.shape[1]):aux_qx.shape[1]])
qx2.rename(columns=lambda x: x.replace('_y', ''), inplace=True)

qx = pd.concat([aux_qx['num_poliza'],qx1 * qx2], axis=1)
qx.columns = ['certificado'] + list(np.arange(0, qx.shape[1]-1))

# Unión de dos bases de datos a través de una llave: cx
caidas = triangle_c.iloc[:,0:(max(rmat['periodo restante'])+1)]
aux_cx = pd.merge(mat_zer2, caidas, on='t', how='left')

cx1 = (aux_cx.iloc[:,2:mat_zer2.shape[1]])
cx1.rename(columns=lambda x: x.replace('_x', ''), inplace=True)
cx2 = (aux_cx.iloc[:,(mat_zer2.shape[1]):aux_cx.shape[1]])
cx2.rename(columns=lambda x: x.replace('_y', ''), inplace=True)

cx = pd.concat([aux_cx['num_poliza'],cx1 * cx2], axis=1)
cx.columns = ['num_poliza'] + list(np.arange(0, cx.shape[1]-1))

4️⃣Construcción de flujos

Ⓐ Parte Nominal

In [ ]:
aux_vigencia = mat_zer.iloc[:, 3:mat_zer.shape[1]]

#-------------------------------------------------------------------------------------------------
# Beneficio
#-------------------------------------------------------------------------------------------------
beneficio = pd.concat([rmat[***]] * max(rmat[***]), axis=1, ignore_index=True)
beneficio = beneficio * aux_vigencia

#-------------------------------------------------------------------------------------------------
# Gastos administrativos
#-------------------------------------------------------------------------------------------------
gastos = pd.concat([rmat[***]] * max(rmat[***]), axis=1, ignore_index=True) * aux_vigencia

#-------------------------------------------------------------------------------------------------
# Primas
#-------------------------------------------------------------------------------------------------
prima = pd.concat([rmat[***]] * max(rmat[***]), axis=1, ignore_index=True) * aux_vigencia

Ⓑ Parte probabilizada

In [ ]:
#-------------------------------------------------------------------------------------------------
# 🌀flujo de sobrevivencia (px)
#-------------------------------------------------------------------------------------------------
mortalidad = qx.iloc[:,1:-1]
caidas = cx.iloc[:,1:cx.shape[1]-1]
px =  (1 - ***) * (1 - ***)

triangle_f = px.copy()
triangle_f = triangle_f.cumprod(axis=1)
triangle_f = pd.DataFrame(triangle_f)

triangle_f.insert(0,'sobrevivencia',1)
cols = np.arange(0,triangle_f.shape[1])
triangle_f.columns = cols

triangle_f = triangle_f * aux_vigencia

#-------------------------------------------------------------------------------------------------
# 🌀flujo de mortalidad (qx)
#-------------------------------------------------------------------------------------------------
triangulo_cobertura = qx.iloc[:,1:] * ***

triangulo_cobertura.insert(0,'cobertura1',0)
cols = np.arange(0, max(rmat['periodo restante'])+1)
triangulo_cobertura.columns = cols

triangulo_cobertura = triangulo_cobertura.iloc[:, :-1] * mat_zer.iloc[:,3:]
triangulo_cobertura.insert(0,'num_poliza', rmat['num_poliza'])
triangulo_cobertura = triangulo_cobertura.iloc[:,0:(max(rmat['periodo restante'])+1)]

#-------------------------------------------------------------------------------------------------
# 🌀flujo de caidas (cx)
#-------------------------------------------------------------------------------------------------
triangulo_caidas = cx.iloc[:,1:cx.shape[1]] * ***

triangulo_caidas.insert(0,'caidas',0)
cols = np.arange(0, max(rmat['periodo restante'])+1)
triangulo_caidas.columns = cols

triangulo_caidas = triangulo_caidas.iloc[:, :-1] * mat_zer2.iloc[:,3:]
triangulo_caidas.insert(0,'num_poliza', rmat['num_poliza'])
triangulo_caidas = triangulo_caidas.iloc[:,0:(max(rmat['periodo restante'])+1)]

Ⓒ Parte financiera

In [ ]:
t = pd.DataFrame(np.arange(0,max(rmat['periodo restante'])))

vector_descuento = pd.DataFrame(vtd[***])
vector_descuento = vector_descuento[0:len(t)]

tasa = pd.DataFrame(np.concatenate([t,vector_descuento],axis=1))
factor_descuento = np.divide(1, pow(1+(tasa[1]),tasa[0]))

mat_factor_descuento = np.transpose(pd.DataFrame(factor_descuento))
mat_factor_descuento = mat_factor_descuento.loc[mat_factor_descuento.index.repeat(rmat.shape[0])].reset_index(drop=True) * aux_vigencia

5️⃣Cálculo de reserva

In [ ]:
flujo_cob = *** * ***
flujo_gastos = *** * ***
flujo_primas = *** * ***

egresos =  *** + ***
ingresos = ***

reserva = (egresos - ingresos) * ***
sumReserva = reserva.sum(axis=1)
sumReserva.sum()

# ☠️ Riesgo de mortalidad
- **Estrés de 15%**

In [ ]:
mat_mortalidad = qx.iloc[:,1:]
mat_mortalidadEM = mat_mortalidad * ***

🔔❗ Nuevo flujo de sobrevivencia (Px)

In [ ]:
mat_caidas = cx.iloc[:,1:]
pxEM =  (1 - ***) * (1-mat_caidas.iloc[:,:-1])

triangle_fEM = pxEM.copy()
triangle_fEM = triangle_fEM.cumprod(axis=1)
triangle_fEM = pd.DataFrame(triangle_fEM)
triangle_fEM.insert(0,'sobrevivencia',1)
cols = np.arange(0,triangle_fEM.shape[1])
triangle_fEM.columns = cols
triangle_fEM = *** * ***

🔔❗ Nuevo flujo de mortalidad (qx)

In [ ]:
triangulo_coberturaEM = *** * ***
triangulo_coberturaEM.insert(0,'cobertura1',0)
cols = np.arange(0, max(rmat['periodo restante'])+1)
triangulo_coberturaEM.columns = cols
triangulo_coberturaEM = triangulo_coberturaEM * mat_zer.iloc[:,3:]

triangulo_coberturaEM.insert(0,'num_poliza', rmat['num_poliza'])
triangulo_coberturaEM = triangulo_coberturaEM.iloc[:,0:(max(rmat['periodo restante'])+1)]

🔔❗ Nuevo flujo de caidas (cx)

In [ ]:
triangulo_caidasEM = *** * ***

triangulo_caidasEM.insert(0,'caidas',0)
cols = np.arange(0, max(rmat['periodo restante'])+1)
triangulo_caidasEM.columns = cols

triangulo_caidasEM = triangulo_caidasEM.iloc[:, :-1] * mat_zer2.iloc[:,3:]
triangulo_caidasEM.insert(0,'num_poliza', rmat['num_poliza'])
triangulo_caidasEM = triangulo_caidasEM.iloc[:,0:(max(rmat['periodo restante'])+1)]

🧮 Flujo de reservas

In [ ]:
flujo_cob1EM = *** * ***
flujo_gastosEM = *** * ***
flujo_primasEM = *** * ***

egresosEM =  *** + ***
ingresosEM = flujo_primasEM
reservaEM = (egresosEM - ingresosEM) * ***
sumReservaEM = reservaEM.sum(axis=1)

🧮 Reserva individual

In [ ]:
reserva_individualEM = reservaEM.sum(axis=1)
reserva_individualEM = pd.DataFrame(reserva_individualEM)
reserva_individualEM[reserva_individualEM < 0] = 0
reserva_individualEM.insert(0, 'num_poliza', rmat['num_poliza'])
reserva_individualEM.rename(columns = {0 : 'Reservas'}, inplace = True)
reserva_individualEM.head()

#📊 Reportería

In [ ]:
matriz_escenarios = pd.concat([***, ***], axis=1, ignore_index=True)
matriz_escenarios.columns = ['modelo', 'mortalidad']
matriz_escenarios[matriz_escenarios < 0] = 0

pd.options.display.float_format = '{:,.2f}'.format
pd.DataFrame(matriz_escenarios.sum(axis=0))

In [ ]:
matriz_riesgos_ =  matriz_escenarios.subtract(matriz_escenarios.iloc[:, 0], axis=0)
matriz_riesgos = matriz_riesgos_[['mortalidad']]

pd.options.display.float_format = '{:,.2f}'.format
pd.DataFrame(matriz_riesgos.sum(axis=0))

In [ ]:
riesgo_mortalidad = matriz_riesgos.sum().squeeze()
riesgo_mortalidad

---
# 📝 3. Agregación
---


In [ ]:
riesgo_longevidad = 160000

matriz_correlacion = pd.DataFrame(np.array([[1, -0.25], [-0.25, 1]]))
matriz_riesgos_vida = pd.DataFrame(np.array([[riesgo_mortalidad], [riesgo_longevidad]]))

# Multiplicacion matricial para el riesgo
riesgo_agregado_vida = np.sqrt(matriz_riesgos_vida.T.dot(matriz_correlacion.dot(matriz_riesgos_vida)))
riesgo_agregado_vida

---
# 4. 📝 Reserva de prima no devengada
---
- Calcule la reserva de prima no devengada de la siguiente cartera

<details>
<summary>💡 Pista 1 (concepto)</summary>

Es el mismo cálculo de RRC = RPND que viste en la **Sesión 4 (Reservas de vida)**, aplicado ahora a esta cartera de 15 pólizas: la **base de cálculo** (prima neta de gastos de adquisición), el **pnc** (proporción de tiempo de cobertura pendiente, acotada entre 0 y 1) y finalmente RPND = base de cálculo × pnc.

<details>
<summary>💡 Pista 2 (código)</summary>

Piensa en: `pd.to_datetime(..., format=...)` para las fechas, `pd.to_numeric(...)` para forzar el tipo numérico en las columnas de montos, la resta entre dos columnas de fechas (te da un `Timedelta`, que puedes dividir entre otro `Timedelta` para obtener un número), y `.clip(lower=0)` para que el pnc no quede negativo en pólizas ya vencidas.

</details>
</details>


In [ ]:
# Nota: primas y gastos de adquisición reescalados (÷10) respecto a la cartera original,
# para que la RPND resultante sea comparable en magnitud con el balance de la Sección 1.
data = {"Número de póliza": ["D01", "D02", "D03", "D04", "D05", "D06", "D07", "D08", "D09", "D10", "D11", "D12", "D13", "D14", "D15"],
    "Riesgo": ["Domiciliario"] * 15,
    "Fecha de inicio de vigencia": ["03/05/2024", "25/03/2024", "31/01/2024", "15/06/2024", "01/04/2024", "15/05/2024", "15/03/2024", "15/02/2024", "15/06/2024", "15/01/2024", "15/02/2024", "15/04/2024", "15/06/2024", "15/08/2024", "15/01/2024"],
    "Fecha de fin de vigencia": ["02/05/2025", "24/03/2025", "30/01/2025", "14/06/2025", "31/03/2025", "14/05/2025", "14/03/2025", "14/02/2025", "14/06/2025", "14/01/2025", "14/02/2025", "14/04/2025", "14/06/2025", "14/08/2025", "14/01/2025"],
    "Prima emitida total neta de coaseguro": [338520, 461125, 338520, 364560, 439425, 565068, 483042, 305102, 342860, 406875, 317362.5, 368900, 428575, 544887, 454289.5],
    "Gastos de adquisición": [67953.2, 59049.7, 59169.5, 53596.9, 63510, 56921.5, 51010.5, 55095.2, 68013.1, 48883, 48587, 55296.5, 58090.8, 71138.2, 64989.1],}

df = pd.DataFrame(data)
df.head(3)

In [ ]:
df.dtypes

➡️ Aplicar formatos

In [ ]:
df['Fecha de inicio de vigencia'] = pd.to_datetime(df['Fecha de inicio de vigencia'], format=***)
df['Fecha de fin de vigencia'] = pd.to_datetime(df['Fecha de fin de vigencia'], format=***)

In [ ]:
df[***] = pd.to_numeric(df[***])
df[***] = pd.to_numeric(df[***])

<h4>📘 Cálculo de la RPND (Reserva para Primas No Devengadas)</h4>

$$
\mathrm{RPND} = \text{Base de cálculo} \times \mathrm{pnc}
$$

<p>
- La <b>Base de Cálculo</b> se define como la <b>prima neta</b> (prima emitida menos devoluciones y anulaciones, si las hubiera).
</p>

<p>
- El porcentaje de prima <b>no devengada (pnc)</b> se calcula como la proporción del tiempo de cobertura pendiente:</p>

$$
\mathrm{pnc} = \frac{\mathrm{FF} - \max(\mathrm{FC}, \mathrm{FI})}{\mathrm{FF} - \mathrm{FI}}
$$

<p>Donde:</p>
<ul>
  <li><b>FF:</b> Fecha de Fin de vigencia</li>
  <li><b>FC:</b> Fecha de Cálculo (por ejemplo, el cierre del mes o del año)</li>
  <li><b>FI:</b> Fecha de Inicio de vigencia</li>
</ul>


In [ ]:
# Calculando la Base de Cálculo
df['Base de Cálculo'] = df['Prima emitida total neta de coaseguro'] - df[***]

# Calculando el pnc
df['pnc'] = (df['Fecha de fin de vigencia'] - fecha_calculo) / (df[***] - df['Fecha de inicio de vigencia'])
df['pnc'] = df['pnc'].clip(lower=***)

# Calculando el RPND
df['rpnd'] = df[***] * df['pnc']

# Sumando toda la columna rpnd
total_rpnd = df['rpnd'].sum()
total_rpnd

> 🔗 **Conexión con el balance:** la RPND que acabas de calcular es, precisamente, la reserva que la compañía registra en la fila **"Reservas de riesgos en curso"** del balance de la Sección 1 — hasta ahora estaba marcada con un guion (`-`) porque todavía no la habíamos calculado. La siguiente celda actualiza esa fila con el valor que obtuviste.
>
> ⚠️ En este ejercicio simplificado los fondos propios están dados; en la práctica, el balance y el requerimiento de capital se recalcularían con estas reservas.

In [ ]:
# Actualizamos el balance con la RPND recién calculada
balance_general.loc[balance_general['Concepto'] == 'Reservas de riesgos en curso', 'Monto ($)'] = total_rpnd
balance_general

---
# 📝 5. Reserva de siniestros



- **Asuma que el triangulo generado ya es acumulado**


---

> 🔗 **Conexión:** El triángulo, los factores de desarrollo y el IBNR se calculan igual que en la Sesión 3 (Triángulos e IBNR).


<details>
<summary>💡 Pista 1 (concepto)</summary>

Mismo flujo que la **Sesión 3 (Triángulos e IBNR)**: construir un `cl.Triangle` acumulado a partir de los datos de siniestros, ajustar los factores de desarrollo con `cl.Development`, proyectar los *ultimates* con `cl.Chainladder`, y obtener el IBNR como la diferencia entre el *ultimate* y lo ya pagado/reportado a la fecha de cálculo.

<details>
<summary>💡 Pista 2 (código)</summary>

Piensa en: `pd.to_datetime(..., format='%d/%m/%Y')` para las columnas de origen y desarrollo, `cl.Triangle(df, origin=..., development=..., columns=..., cumulative=True, ...)`, `cl.Development(average=...).fit_transform(triangulo)`, `cl.Chainladder().fit(triangulo_desarrollado)`, y los atributos `.ultimate_` / `.ibnr_` del objeto ya ajustado.

</details>
</details>


In [ ]:
data = {
    "Ocurrencia": ["30/06/2016", "30/06/2016", "30/06/2016", "30/06/2016", "30/06/2016", "30/06/2016", "30/06/2016", "30/06/2016",
                   "30/06/2017", "30/06/2017", "30/06/2017", "30/06/2017", "30/06/2017", "30/06/2017", "30/06/2017",
                   "30/06/2018", "30/06/2018", "30/06/2018", "30/06/2018", "30/06/2018", "30/06/2018",
                   "30/06/2019", "30/06/2019", "30/06/2019", "30/06/2019", "30/06/2019",
                   "30/06/2020", "30/06/2020", "30/06/2020", "30/06/2020",
                   "30/06/2021", "30/06/2021", "30/06/2021",
                   "30/06/2022", "30/06/2022",
                   "30/06/2023"],
    "Movimiento": ["30/06/2016", "30/06/2017", "30/06/2018", "30/06/2019", "30/06/2020", "30/06/2021", "30/06/2022", "30/06/2023",
                   "30/06/2017", "30/06/2018", "30/06/2019", "30/06/2020", "30/06/2021", "30/06/2022", "30/06/2023",
                   "30/06/2018", "30/06/2019", "30/06/2020", "30/06/2021", "30/06/2022", "30/06/2023",
                   "30/06/2019", "30/06/2020", "30/06/2021", "30/06/2022", "30/06/2023",
                   "30/06/2020", "30/06/2021", "30/06/2022", "30/06/2023",
                   "30/06/2021", "30/06/2022", "30/06/2023",
                   "30/06/2022", "30/06/2023",
                   "30/06/2023"],
    "Monto": [4866866, 5623362, 5956121, 6024934, 6138520, 6158403, 6159925, 6159925,
              4969162, 5692039, 6092048, 6304998, 6387073, 6408356, 6414333,
              4957403, 5765104, 5997770, 6200373, 6234124, 6252633,
              4654736, 5366563, 5743215, 5939062, 6016303,
              4866866, 5645829, 5975051, 6247129,
              4831289, 5451486, 5783333,
              4857955, 6327873,
              3594213]
}


siniestros = pd.DataFrame(data)

In [ ]:
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# Formatear las fechas del dataframe
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
siniestros['***'] = pd.to_datetime(siniestros['***'],  format='%d/%m/%Y')
siniestros['***'] = pd.to_datetime(siniestros['***'], format='%d/%m/%Y')

#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# Formatear las fechas del dataframe
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
cumTri = cl.Triangle(
    siniestros,
    origin="***",
    development="***",
    columns="***",
    cumulative = True,
    origin_format='%d/%m/%Y',
    development_format='%d/%m/%Y',
    trailing=False)

In [ ]:
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# Generamos un objeto "desarrollo" de la biblioteca ChainLadder
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
triangulo = cl.Development(average="volume").fit_transform(cumTri)

In [ ]:
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# Mostrar los factores de desarrollo incrementales y acumulados
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
triangulo.ldf_

In [ ]:
triangulo.cdf_

In [ ]:
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# Creamos un objeto ChainLadder con el cual podamos proyectar los ultimates y obtener la reserva IBNR
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
chain_ladder = cl.Chainladder().fit(***)

In [ ]:
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# Calculamos los ultimates:
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
ultimate_cl = chain_ladder.***
ultimate_cl

### **Calculamos el IBNR:**

In [ ]:
# IBNR
ibnr = chain_ladder.***
print("Suma del IBNR:", "{:,.1f}".format(ibnr.sum()))
ibnr

---
# 📝 6. Riesgo de reserva
---

> 🔗 **Conexión:** El bootstrap ODP sobre el triángulo es el mismo procedimiento de la Sesión 6 (Riesgo de reserva, cópulas y solvencia).


### **Estime el riesgo de reserva asociado:**

<details>
<summary>💡 Pista 1 (concepto)</summary>

Igual que en la **Sesión 6 (Riesgo de reserva y solvencia)**: simulas miles de triángulos con bootstrap ODP a partir del triángulo original, obtienes una distribución de *ultimates* simulados, y comparas el VaR/TVaR al 99% contra el promedio para obtener el riesgo de reserva.

<details>
<summary>💡 Pista 2 (código)</summary>

Piensa en: `cl.BootstrapODPSample(n_sims=..., random_state=...).fit(triangulo).resample_`, `cl.Chainladder().fit(muestras).ultimate_.to_frame()`, sumar por fila con `.sum(axis=1)`, `np.percentile(array, 99)` para el VaR, y para el TVaR filtrar los valores `>= VaR_99` y promediarlos con `.mean()`.

</details>
</details>


In [ ]:
# Bootstrapping

samples = (cl.BootstrapODPSample(n_sims=10000,random_state=5).fit(***).***)
samples

In [ ]:
ultimates_bootstrap = cl.Chainladder().fit(***).***.to_frame()
ultimates_bootstrap

In [ ]:
ultimates_bootstrap['Suma_Ultimates'] = ultimates_bootstrap.***(axis=1)
ultimates_bootstrap

In [ ]:
sorted_losses = np.sort(ultimates_bootstrap['Suma_Ultimates'])

VaR_99 = np.percentile(***, 99)

tvar_99 = sorted_losses[sorted_losses >= ***].***()

promedio_reserva = sorted_losses.mean()

riesgo_reserva = tvar_99 - promedio_reserva

print(f"Var al 99%: {VaR_99:,.0f}")
print(f"TVaR al 99%: {tvar_99:,.0f}")
print(f"Promedio: {promedio_reserva:,.0f}")
print(f"El riesgo es: {riesgo_reserva:,.0f}")

---
# 📝 7. Riesgo de prima
---

> 🔗 **Conexión:** El ajuste de distribuciones (lognormal/gamma) y el cálculo de VaR/TVaR son los de la Sesión 5 (Simulación y distribuciones de pérdida).


### **Primero obtenemos los ratios de siniestralidad**

<details>
<summary>💡 Pista 1 (concepto)</summary>

Como en la **Sesión 5 (Simulación y distribuciones de pérdida)**: calculas ratios de siniestralidad históricos (*ultimate* / prima esperada), ajustas una distribución (lognormal o gamma) a esos ratios, simulas miles de ratios con esa distribución, y calculas el riesgo de prima como (TVaR al 99% − promedio) aplicado sobre la prima emitida.

<details>
<summary>💡 Pista 2 (código)</summary>

Piensa en: `chain_ladder.ultimate_` para los *ultimates*, `np.percentile(...)` para el VaR, filtrar los valores por encima del VaR y usar `.mean()` para el TVaR, y multiplicar el riesgo (en proporción) por `prima_emitida` para pasarlo a monto.

</details>
</details>


In [ ]:
primas = [51135019, 66860873, 86807933, 69632806,88179308, 83236667, 82256835, 61591110]
expected_loss_apriori = chain_ladder.*** * 0 + pd.DataFrame(primas)
expected_loss_apriori

In [ ]:
lr = ultimate_cl/expected_loss_apriori
lr

In [ ]:
data = lr.values.flatten()

n = 10000

##########################################################################################################################
# Log-normal

shape, loc, scale = stats.lognorm.fit(data, floc=0)

#Parámetros
mu = np.log(scale)  # Mu para la Log-normal

sigma = shape  # Sigma para la Log-normal

##########################################################################################################################
# Gamma

#Parámetros
alpha, loc, theta = stats.gamma.fit(data, floc=0)

##########################################################################################################################


# Calibración con Kolmogorov-Smirnov

ks_lognorm = stats.kstest(data, 'lognorm', args=(sigma, 0, np.exp(mu)))
ks_gamma = stats.kstest(data, 'gamma', args=(alpha, 0, theta))


# Comparar y seleccionar la mejor distribución

if ks_lognorm.statistic < ks_gamma.statistic:
    print("La distribución Log-Normal es un mejor ajuste.")
    best_fit = 'lognorm'
    params = (sigma, 0, np.exp(mu))
else:
    print("La distribución Gamma es un mejor ajuste.")
    best_fit = 'gamma'
    params = (alpha, 0, theta)


##########################################################################################################################

# Simular n ratios de siniestralidad (random_state fijo para reproducibilidad)

n = 10000  # Define la cantidad de datos a simular

if best_fit == 'lognorm':
    simulated_data = stats.lognorm.rvs(*params, size=n, random_state=42)
else:
    simulated_data = stats.gamma.rvs(*params, size=n, random_state=42)


print(f"KS Statistic Log-Normal: {ks_lognorm.statistic:.4f}")
print(f"KS Statistic Gamma: {ks_gamma.statistic:.4f}")

In [ ]:
ratios = pd.DataFrame({'Ratios Simulados':simulated_data})
ratios

In [ ]:
sorted_ratios = np.sort(ratios['Ratios Simulados'] )

VaR_99 = np.***(sorted_ratios, 99)

tvar_99 = sorted_ratios[sorted_ratios >= ***].***()

promedio_prima = sorted_ratios.***()

riesgo = tvar_99 - promedio_prima

riesgo

In [ ]:
prima_emitida = 80000000

riesgo_prima = *** * ***

riesgo_prima

---
# 📝 8. Agregación de riesgos
---


<details>
<summary>💡 Pista 1 (concepto)</summary>

Como en la **Sesión 6 (Agregación de riesgos)**: armas un vector con los riesgos individuales (primero reserva y prima; después vida, no vida, mercado, contraparte y operacional), lo combinas con una matriz de correlación, y agregas con la fórmula raíz(vector ᵀ · matriz_correlación · vector) para obtener el riesgo diversificado. El requerimiento de capital es la suma del riesgo agregado más el riesgo operacional, y el ratio de solvencia es fondos propios / requerimiento.

<details>
<summary>💡 Pista 2 (código)</summary>

Piensa en: `pd.DataFrame(np.array([[...], [...]]))` para armar el vector columna de riesgos, `np.sqrt(vector.T.dot(matriz.dot(vector)))` para la raíz de la forma cuadrática, y una simple división para el ratio de solvencia (`rsol = fondos_propios / requerimiento`).

</details>
</details>


### **Calcule el riesgo agregado de reserva y prima:**

In [ ]:
# Agregación

correlation_matrix = pd.DataFrame(np.array([
    [1, 0.5],
    [0.5, 1]]))

matriz_riesgos_no_vida = pd.DataFrame(np.array([
    [***],
    [***]]))

# Multiplicacion matricial para el riesgo
riesgo_agregado_no_vida = np.sqrt(***.T.dot(correlation_matrix.dot(***)))
riesgo_agregado_no_vida

---
# 📝 9. Agregación final
---
*Nota: `riesgo_vida` y `riesgo_no_vida` se obtienen de `riesgo_agregado_vida` (sección 3, riesgo de mortalidad + longevidad) y de `riesgo_agregado_no_vida` (agregación de riesgo de prima y de reserva calculada arriba), por lo que el resultado ya es consistente con lo calculado en el notebook y no depende de valores fijos arbitrarios.*

In [ ]:
# Matriz de correlacion
correlation_matrix = pd.DataFrame(np.array([
    [1.00, 0.25, 0.50, 0.25],
    [0.25, 1.00, 0.25, 0.25],
    [0.25, 0.25, 1.00, 0.25],
    [0.25, 0.25, 0.25, 1.00]
]))

riesgo_vida = ***

riesgo_no_vida = ***

riesgo_mercado = 1500000

riesgo_contraparte = 300000

riesgo_operacional = 1200000

fondos_propios =  18714417

### **Estime el riesgo agregado**

In [ ]:
matriz_riesgos = pd.DataFrame(np.array([
    [***],
    [***],
    [***],
    [***],
]))

matriz_riesgos

In [ ]:
riesgo_agregado = np.sqrt(***.T.dot(***.dot(***)))
riesgo_agregado

In [ ]:
requerimiento = *** + ***
requerimiento

### **Estime el requerimiento de capital**

In [ ]:
rsol = ***/***
rsol